# Giai đoạn 1 – Dữ liệu công khai CNN/DailyMail và EDA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23010216-hieusphamj/text-summarization-nlp/blob/main/notebooks/01_eda_cnn_dm.ipynb)

**Mục tiêu**
1. Tải CNN/DailyMail và tạo tập con có seed cố định (20k train / 1k val / 1k test).
2. Phân tích: độ dài bài và tóm tắt, tỉ lệ nén, mức độ *abstractive* (n-gram mới), *lead bias*, và "rác" trong văn bản.
3. Kết quả EDA là **bằng chứng** cho `docs/decisions.md` (D2) và là cơ sở cho bước làm sạch ở Giai đoạn 3.

> Chạy từ trên xuống dưới (Runtime → Run all). Không cần GPU ở giai đoạn này.

In [ ]:
# 0. Thiết lập môi trường (tự nhận biết Colab hay máy cục bộ)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/23010216-hieusphamj/text-summarization-nlp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("src").exists() and not Path("text-summarization-nlp").exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)
if Path("text-summarization-nlp").exists():
    os.chdir("text-summarization-nlp")
while not Path("src").exists() and Path.cwd() != Path.cwd().parent:   # chạy từ thư mục notebooks/
    os.chdir("..")
sys.path.insert(0, os.getcwd())

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

print("Thư mục làm việc:", os.getcwd())

In [ ]:
# 1. Tải dữ liệu và tạo tập con (bỏ qua nếu đã có)
from src.data.cnn_dm import main as build_data

if not Path("data/processed/cnn_dm_train.jsonl").exists():
    build_data("configs/data_cnn_dm.yaml")
else:
    print("Đã có data/processed/, bỏ qua bước tải.")

In [ ]:
# 2. Nạp dữ liệu
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data import eda_utils as u

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
FIG = Path("results/figures"); FIG.mkdir(parents=True, exist_ok=True)
TAB = Path("results/tables");  TAB.mkdir(parents=True, exist_ok=True)

data = {s: pd.read_json(f"data/processed/cnn_dm_{s}.jsonl", lines=True)
        for s in ["train", "validation", "test"]}
train = data["train"]

sizes = pd.read_csv(TAB / "cnn_dm_split_sizes.csv") if (TAB / "cnn_dm_split_sizes.csv").exists() else None
display(sizes) if sizes is not None else print({k: len(v) for k, v in data.items()})
train.head(2)

## 1. Độ dài bài viết và bản tóm tắt

In [ ]:
train["art_words"] = train["article"].map(u.word_count)
train["sum_words"] = train["highlights"].map(u.word_count)
train["sum_sents"] = train["highlights"].map(lambda t: len(u.summary_sentences(t)))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
a_clip = train["art_words"].clip(upper=train["art_words"].quantile(0.99))
ax[0].hist(a_clip, bins=50, color="#4a6fa5"); ax[0].set_title("Độ dài bài viết (số từ, cắt ở p99)")
ax[0].axvline(400, ls="--", c="r"); ax[0].text(400, ax[0].get_ylim()[1]*0.9, " ~512 token", color="r")
ax[1].hist(train["sum_words"], bins=40, color="#3c8d53"); ax[1].set_title("Độ dài tóm tắt (số từ)")
for a in ax: a.set_xlabel("số từ"); a.set_ylabel("số mẫu")
plt.tight_layout(); plt.savefig(FIG / "cnn_dm_length_hist.png", dpi=200); plt.show()

len_tbl = pd.DataFrame({"article_words": u.describe(train["art_words"]),
                        "summary_words": u.describe(train["sum_words"]),
                        "summary_sentences": u.describe(train["sum_sents"])}).round(1)
len_tbl

## 2. Tỉ lệ nén (số từ bài / số từ tóm tắt)

In [ ]:
train["compression"] = [u.compression_ratio(a, s) for a, s in zip(train["article"], train["highlights"])]
c = train["compression"]
plt.figure(figsize=(6, 3.6))
plt.hist(c.clip(upper=c.quantile(0.99)), bins=50, color="#c9a227")
plt.title("Tỉ lệ nén bài : tóm tắt"); plt.xlabel("lần"); plt.ylabel("số mẫu")
plt.tight_layout(); plt.savefig(FIG / "cnn_dm_compression.png", dpi=200); plt.show()
print(f"Trung vị tỉ lệ nén: {c.median():.1f}x")

## 3. Mức độ abstractive: tỉ lệ n-gram của tóm tắt **không** có trong bài
Giá trị càng thấp thì tóm tắt càng "chép lại" bài (thiên về extractive).

In [ ]:
N = min(5000, len(train))
sub = train.sample(N, random_state=42)
novel = {n: [u.novel_ngram_ratio(a, s, n) for a, s in zip(sub["article"], sub["highlights"])] for n in (1, 2, 3)}
novel_mean = {f"{n}-gram": float(np.nanmean(v)) for n, v in novel.items()}

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].bar(novel_mean.keys(), [v*100 for v in novel_mean.values()], color=["#4a6fa5", "#3c8d53", "#c0504d"])
ax[0].set_ylabel("% n-gram mới"); ax[0].set_title("Tỉ lệ n-gram mới trong tóm tắt")
for i, v in enumerate(novel_mean.values()): ax[0].text(i, v*100 + 1, f"{v*100:.1f}%", ha="center")
ax[1].hist(np.array(novel[2])[~np.isnan(novel[2])], bins=30, color="#3c8d53")
ax[1].set_title("Phân bố tỉ lệ bigram mới (mỗi mẫu)"); ax[1].set_xlabel("tỉ lệ")
plt.tight_layout(); plt.savefig(FIG / "cnn_dm_novel_ngrams.png", dpi=200); plt.show()
novel_mean

## 4. Lead bias: nội dung tóm tắt nằm ở đâu trong bài?
Với mỗi câu tóm tắt, tìm câu trong bài có nhiều từ trùng nhất rồi thống kê vị trí câu đó.

In [ ]:
M = min(2000, len(train))
pos = []
for a, h in zip(train["article"].head(M), train["highlights"].head(M)):
    pos += u.best_match_positions(a, h)
pos = np.array(pos)
within3 = float((pos < 3).mean()); within10 = float((pos < 10).mean())

cap = 15
counts = np.bincount(np.minimum(pos, cap), minlength=cap + 1)
plt.figure(figsize=(7, 3.6))
plt.bar(range(cap + 1), counts / counts.sum() * 100, color="#4a6fa5")
plt.xticks(range(cap + 1), [str(i + 1) if i < cap else f"{cap + 1}+" for i in range(cap + 1)])
plt.xlabel("vị trí câu trong bài (1 = câu đầu)"); plt.ylabel("% câu tóm tắt")
plt.title(f"Vị trí câu nguồn của tóm tắt (3 câu đầu: {within3*100:.0f}%)")
plt.tight_layout(); plt.savefig(FIG / "cnn_dm_lead_bias.png", dpi=200); plt.show()
print(f"Nằm trong 3 câu đầu: {within3*100:.1f}% | trong 10 câu đầu: {within10*100:.1f}%")

## 5. Văn bản chứa 'rác' cần làm sạch (động lực cho Giai đoạn 3)

In [ ]:
flags = pd.DataFrame([u.boilerplate_flags(a) for a in train["article"]])
bp = (flags.mean() * 100).round(2).rename("% bài chứa").to_frame()
bp

In [ ]:
# Xem vài ví dụ đầu bài để nhận diện mẫu rác
for a in train["article"].sample(4, random_state=1):
    print(repr(a[:220])); print("---")

## 6. Lưu bảng tổng hợp

In [ ]:
summary = pd.DataFrame([
    ("Số mẫu train/val/test (tập con)", " / ".join(str(len(data[s])) for s in data)),
    ("Số từ bài – trung vị", round(train["art_words"].median(), 1)),
    ("Số từ bài – p95", round(train["art_words"].quantile(0.95), 1)),
    ("Số từ tóm tắt – trung vị", round(train["sum_words"].median(), 1)),
    ("Số câu tóm tắt – trung bình", round(train["sum_sents"].mean(), 2)),
    ("Tỉ lệ nén – trung vị (x)", round(train["compression"].median(), 1)),
    ("% unigram mới", round(novel_mean["1-gram"] * 100, 1)),
    ("% bigram mới", round(novel_mean["2-gram"] * 100, 1)),
    ("% trigram mới", round(novel_mean["3-gram"] * 100, 1)),
    ("% câu tóm tắt có nguồn ở 3 câu đầu", round(within3 * 100, 1)),
    ("% bài chứa dấu vết rác (bất kỳ mẫu nào)", round(float(flags.any(axis=1).mean() * 100), 1)),
], columns=["chỉ số", "giá trị"])
summary.to_csv(TAB / "cnn_dm_eda_summary.csv", index=False)
summary

## 7. Nhận xét (điền sau khi chạy xong, rồi chép vào `docs/decisions.md` mục D2)

- **Độ dài:** bài trung vị ... từ, tóm tắt trung vị ... từ → chọn `max_input = 512`, `max_target = 128` vì ...
- **Abstractive hay extractive:** ...% bigram của tóm tắt không có trong bài → dữ liệu thiên về ...
- **Lead bias:** ...% câu tóm tắt lấy từ 3 câu đầu → Lead-3 sẽ là baseline mạnh, cần so sánh công bằng.
- **Rác dữ liệu:** ...% bài có tiền tố/bản quyền → cần làm sạch ở Giai đoạn 3.

In [ ]:
# 8. (Colab) Đóng gói kết quả để tải về và commit vào repo
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("phase1_outputs", "zip", ".", "results")
    shutil.copy("data/samples/cnn_dm_sample.jsonl", "cnn_dm_sample.jsonl")
    files.download("phase1_outputs.zip"); files.download("cnn_dm_sample.jsonl")